# HistoVision — обучение сегментационной модели на BCSS (Colab GPU)

Прогоняется через официальное расширение **Colab для VS Code** (Kernel → Select Kernel → Colab → New Colab Server → GPU/T4), либо напрямую в браузере на colab.research.google.com.

**Важно:** в списке ядер VS Code может быть несколько вариантов (в т.ч. Julia, если он у вас установлен локально) — убедитесь, что выбран именно **Python 3** / Colab-рантайм, а не Julia. Признак, что ядро не то: `!nvidia-smi` в первой ячейке падает с `UndefVarError` — это Julia, а не Python, пытается выполнить строку как код.

**Если ячейка 1 показывает `nvidia-smi: command not found` / `CUDA available: False`, хотя вы выбирали GPU/T4** — значит подключился CPU-рантайм. "Restart" в VS Code перезапускает только Python-процесс, а не саму Colab-машину, поэтому если раньше уже был создан CPU-сеанс, "Restart" к нему и переподключит. Решение: через кнопку выбора ядра явно создайте **новый** Colab-сеанс (New Colab Server) с GPU, а не переиспользуйте существующий.

Код тянется из GitHub-репозитория, ветка `feature/segmentation` — URL уже прописан во второй code-ячейке. Ячейка 2 сама разбирается, клонировать репозиторий с нуля или обновить существующий (`git pull`) — безопасно перезапускать.

**Почему прошлый прогон "завис" на много часов без вывода (уже исправлено):** `--stain-normalize` запускал нормализацию окраски Macenko на *целом* BCSS-регионе (некоторые — свыше 100 млн пикселей) перед вырезанием обучающего патча, а не после — вместо долей секунды на патч это отнимало секунды-минуты на каждое изображение, на каждой эпохе, без единой строчки прогресса в логе. Оба узких места исправлены: нормализация теперь идёт по уже вырезанному 256x256 патчу (быстро, и так же, как на инференсе), и появилось логирование после каждого батча/эпохи — медленный прогон и зависший теперь визуально не спутать.

**Устойчивость к обрыву сеанса (важно на бесплатном Colab — сессия может оборваться сама по себе, максимум ~12 часов, отключение при простое, GPU не гарантирован в пик):**
- Ячейка 3 монтирует Google Drive — постоянное хранилище для датасета/чекпоинтов/логов, переживающее даже полную пересборку Colab-машины (не только перезапуск Python-процесса, как /content и так переживает).
- Ячейка 5 (скачивание BCSS) качает напрямую с официального сервера датасета (Girder REST API + Figshare), не с Google Drive — без дневной квоты, которая раньше обрывала докачку на середине; резюмируемо. Само обучение читает данные с локального диска (Drive как сетевой FUSE-диск заметно медленнее для многократного построчного чтения) — ячейка 5 синхронизирует копии в обе стороны.
- Ячейка 6 (обучение) пишет полное состояние (веса + оптимизатор + номер эпохи) на Drive после **каждой** эпохи и подхватывает его при перезапуске — если сеанс оборвётся, просто запустите ячейку 6 заново. Останавливается само по плато val mIoU (`--patience`), не по жёстко заданному числу эпох.
- Ячейка 7 показывает, на какой эпохе остановились и какой mIoU был лучшим — но её можно запустить только пока ячейка 6 не выполняется (один кернел = одна ячейка за раз): либо после её завершения, либо прервав её (Interrupt, не Restart) — resume.pt переживёт это прерывание.

**Что делает этот ноутбук:**
1. Проверяет GPU-рантайм.
2. Клонирует репозиторий и ставит зависимости.
3. Монтирует Google Drive, заводит постоянные пути (датасет/чекпоинты/логи).
4. Ставит зависимости (включая `girder-client` для скачивания BCSS).
5. Скачивает и готовит полный датасет BCSS, 151 регион (`prepare_bcss.py download-official`), синхронизирует с Drive.
6. Обучает DeepLabV3+/ResNet-18 (`train.py`) на GPU, с возобновлением, ранней остановкой по плато и подробным логированием по батчам/эпохам.
7. Печатает статус обучения (эпоха, лучший mIoU, история) — только пока ячейка 6 не выполняется.
8. Визуализирует маску сегментации, наложенную на 2-3 препарата из валидации, сохраняет превью на Drive.


In [1]:
# 1. Проверка GPU
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (нет GPU-рантайма — выберите GPU в Select Kernel)")

Mon Aug 24 04:19:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Клонирование репозитория (или git pull, если /content уже содержит клон
# с прошлого запуска — VS Code "Restart" перезапускает только Python-процесс,
# не саму Colab-машину, так что файлы в /content переживают рестарт).
# git pull трогает только код из репозитория; data/bcss — вне git
# (gitignored), поэтому уже скачанные файлы при квоте Google Drive не
# теряются между повторными запусками.
#
# Если REPO_DIR существует, но не является валидным git-репозиторием
# (например, прошлый clone прервался на середине) — просто git clone в
# него упадёт с "already exists". В этом случае бэкапим data/bcss (если
# есть), стираем директорию и клонируем заново, восстанавливая данные.
import os
import shutil

REPO_URL = "https://github.com/soon00sad/histo_learn.git"
BRANCH = "feature/segmentation"
REPO_DIR = "/content/HistoVision"
BACKUP_DIR = "/content/_bcss_backup"

if os.path.isdir(f"{REPO_DIR}/.git"):
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
else:
    if os.path.isdir(REPO_DIR):
        data_dir = f"{REPO_DIR}/data/bcss"
        if os.path.isdir(data_dir):
            print(f"{REPO_DIR} существует, но это не валидный git-репозиторий — "
                  f"бэкаплю {data_dir} перед пересозданием.")
            shutil.rmtree(BACKUP_DIR, ignore_errors=True)
            shutil.move(data_dir, BACKUP_DIR)
        shutil.rmtree(REPO_DIR)
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
    if os.path.isdir(BACKUP_DIR):
        os.makedirs(f"{REPO_DIR}/data", exist_ok=True)
        shutil.move(BACKUP_DIR, f"{REPO_DIR}/data/bcss")
        print("data/bcss восстановлена из бэкапа.")

%cd {REPO_DIR}

Cloning into '/content/HistoVision'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (259/259), done.
remote: Compressing objects: 100% (171/171), done.
remote: Total 259 (delta 108), reused 231 (delta 80), pack-reused 0 (from 0)
Receiving objects: 100% (259/259), 744.00 KiB | 22.54 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/HistoVision


In [ ]:
# 3. Смонтировать Google Drive и завести на нём постоянные пути для датасета,
# чекпоинтов и логов. Это переживает не только перезапуск Python-процесса
# (как /content уже переживает благодаря ячейке 2), но и полную пересборку
# самой Colab-машины (бесплатный тариф может отозвать её в любой момент,
# особенно после ~12 часов или простоя) — без этого шага и скачанный
# датасет, и прогресс обучения терялись бы при таком обрыве.
#
# Датасет НЕ читается напрямую с Drive во время обучения (ячейка 5 копирует
# его на локальный диск /content перед обучением, и синхронизирует обратно
# после) — Drive смонтирован как сетевой FUSE-диск, и построчное чтение
# больших PNG (некоторые BCSS-регионы — свыше 100 млн пикселей) на каждой
# эпохе через него заметно медленнее локального SSD. Drive здесь —
# персистентное хранилище на случай обрыва сеанса, не рабочая копия для
# самого обучения.
#
# При первом запуске drive.mount() покажет ссылку для авторизации — перейдите
# по ней и вставьте код в поле, которое появится. Если авторизация не
# проходит через VS Code (MessageError: User cancelled ...) — откройте
# colab.research.google.com в браузере, Runtime -> Manage sessions, найдите
# этот же GPU-сеанс и выполните там именно эту ячейку (mount) один раз;
# дальше можно продолжать в VS Code — это тот же бэкенд.
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/histovision")
DRIVE_DATA_DIR = DRIVE_ROOT / "data" / "bcss"
DRIVE_CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
DRIVE_LOG_DIR = DRIVE_ROOT / "logs"
for d in (DRIVE_DATA_DIR, DRIVE_CHECKPOINT_DIR, DRIVE_LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)


def sync_dir(src: Path, dst: Path) -> None:
    """One-way, skip-if-exists file copy (recursive) — used both directions
    between Drive and local /content in cell 5. Deliberately not
    shutil.copytree (which overwrites every existing file on every call):
    with 151 multi-megapixel region files, re-copying ones already present
    would waste minutes on every single re-run of the download cell."""
    import shutil

    if not src.exists():
        return
    dst.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        target = dst / item.name
        if item.is_dir():
            sync_dir(item, target)
        elif not target.exists():
            shutil.copy2(item, target)


print("Датасет на Drive (персистентное хранилище): ", DRIVE_DATA_DIR)
print("Чекпоинты:                                  ", DRIVE_CHECKPOINT_DIR)
print("Логи:                                        ", DRIVE_LOG_DIR)


In [5]:
# 4. Зависимости. torch/torchvision и большинство ML-пакетов в Colab уже стоят —
# их не трогаем. pydantic/PyYAML НЕ пинуем здесь (в отличие от requirements.txt
# основного проекта) — жёсткая версия конфликтует с уже установленными в Colab
# пакетами (google-genai/google-adk/albumentations/fastapi требуют более новый
# pydantic); наш src/utils/bcss_classes.py и config.py не используют ничего
# version-specific, так что более новый pydantic/PyYAML тут безопасен.
# girder-client — для скачивания BCSS с официального сервера (ячейка 5), без
# квоты Google Drive, которая раньше обрывала докачку на середине; gdown
# остаётся как fallback-путь (prepare_bcss.py download), на всякий случай.
!pip install -q segmentation-models-pytorch==0.5.0 girder-client==5.0.16 gdown==6.1.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00


In [ ]:
# 5. Подготовка BCSS — полный датасет (все 151 регион) с официального
# источника (Girder REST API демо-сервера HistomicsTK + прямые ссылки на
# Figshare для масок), а не с Google Drive — без дневной квоты на
# скачивание, которая раньше обрывала докачку на середине.
#
# Порядок: сначала быстро копируем на локальный диск то, что уже есть на
# Drive с прошлого сеанса (если есть) — так при повторной пересборке
# Colab-машины не нужно заново качать всё по сети. Затем докачиваем
# недостающее с официального источника (резюмируемо — уже присутствующие
# пары пропускаются). В конце синхронизируем локальную копию обратно на
# Drive, чтобы следующий сеанс уже не тянул из сети то, что скачано сейчас.
# Само обучение (ячейка 6) читает данные ТОЛЬКО с локального диска — Drive
# как сетевой FUSE-диск заметно медленнее для многократного построчного
# чтения больших файлов на каждой эпохе (см. ячейку 3).
import os
import shutil
from pathlib import Path

local_data_dir = Path("data/bcss")
if local_data_dir.is_symlink():  # from an older version of this notebook
    local_data_dir.unlink()
local_data_dir.mkdir(parents=True, exist_ok=True)

print("Копирую уже скачанное с Drive на локальный диск (если есть)...")
sync_dir(DRIVE_DATA_DIR, local_data_dir)

!python -m src.training.prepare_bcss download-official --out data/bcss

n_images = len(os.listdir("data/bcss/images")) if os.path.isdir("data/bcss/images") else 0
n_masks = len(os.listdir("data/bcss/masks")) if os.path.isdir("data/bcss/masks") else 0
print(f"Скачано изображений: {n_images}, масок: {n_masks} (ожидается 151/151).")
if n_images < 151 or n_masks < 151:
    print("Меньше 151 — часть регионов не скачалась (см. предупреждения выше, "
          "часто временная сетевая ошибка). Запустите эту ячейку ещё раз: уже "
          "скачанные пары не перекачиваются, докачаются только недостающие.")

print("Синхронизирую локальную копию обратно на Drive (на случай обрыва сеанса)...")
sync_dir(local_data_dir, DRIVE_DATA_DIR)
print("Готово.")


In [ ]:
# 6. Обучение DeepLabV3+/ResNet-18 на GPU. Чекпоинт с лучшим val mIoU
# (--out), полное состояние для возобновления (--resume-path: веса +
# оптимизатор + номер эпохи + история) и лог по эпохам (--history-jsonl)
# пишутся прямо на Drive — если сеанс Colab оборвётся, просто запустите эту
# ячейку заново: она подхватит последнюю сохранённую эпоху и продолжит, а
# не начнёт обучение с нуля.
#
# Прогресс печатается после каждого батча и каждой эпохи (--log-every-batches,
# по умолчанию 1) — раньше здесь не было вообще никакого вывода между стартом
# и концом эпохи, из-за чего медленный прогон было не отличить от зависшего.
#
# --patience/--min-epochs — ранняя остановка по плато val mIoU вместо
# фиксированного числа эпох (--epochs — лишь верхняя граница на всякий
# случай, не цель). --stain-normalize включает ту же нормализацию Macenko,
# что и на инференсе, теперь применяется к уже вырезанному 256x256 патчу
# (как и на инференсе — Macenko всегда идёт по тайлу, не по целому
# препарату), а не к полному BCSS-региону (который бывает больше 100 млн
# пикселей) — раньше именно это составляло реальное узкое место. Веса
# энкодера ResNet-18 — ImageNet (encoder_weights="imagenet" по умолчанию) —
# критично на датасете такого размера (151 регион).
#
# --num-workers параллелит чтение/декодирование файлов с диска в отдельных
# процессах, пока GPU считает текущий батч — на бесплатном Colab обычно
# доступно 2 vCPU.
!python -m src.training.train \
    --data-dir data/bcss \
    --out /content/drive/MyDrive/histovision/checkpoints/segmentation_best.pth \
    --resume-path /content/drive/MyDrive/histovision/checkpoints/resume.pt \
    --history-jsonl /content/drive/MyDrive/histovision/logs/history.jsonl \
    --encoder resnet18 \
    --epochs 100 \
    --patience 10 \
    --min-epochs 8 \
    --batch-size 16 \
    --crop-size 256 \
    --num-workers 2 \
    --stain-normalize \
    --device cuda


In [ ]:
# 7. Отчёт о состоянии обучения. ВАЖНО: в один момент времени ноутбук
# выполняет только одну ячейку — пока ячейка 6 (`!python -m ...`) держит
# процесс, эта ячейка физически не может начать выполняться, а просто ждёт
# своей очереди (это не баг именно этой ячейки — так работает любой
# Jupyter/Colab-кернел). Чтобы посмотреть статус ПОКА обучение идёт —
# прервите выполнение ячейки 6 (кнопка Interrupt/Stop, не Restart) и
# запустите эту; прогресс не потеряется — resume.pt сохраняется после
# каждой эпохи, и повторный запуск ячейки 6 продолжит именно с этого места.
# Если ячейка 6 уже завершилась (сама или по ошибке) — просто запустите эту
# ячейку как обычно.
import json
from pathlib import Path

import torch

ckpt_path = Path("/content/drive/MyDrive/histovision/checkpoints/segmentation_best.pth")
resume_path = Path("/content/drive/MyDrive/histovision/checkpoints/resume.pt")
history_path = Path("/content/drive/MyDrive/histovision/logs/history.jsonl")

print("Лучший чекпоинт:", ckpt_path)
print("  существует:", ckpt_path.exists())
if ckpt_path.exists():
    print("  размер (МБ):", round(ckpt_path.stat().st_size / 1e6, 1))

if resume_path.exists():
    state = torch.load(resume_path, map_location="cpu")
    print(f"\nПоследняя завершённая эпоха: {state['epoch']}")
    print(f"Лучший val mIoU: {state['best_miou']:.4f}")
    print(f"Эпох без улучшения подряд: {state['epochs_without_improvement']}")
else:
    print("\nresume.pt ещё не создан — ячейка 6 либо не запускалась, либо не "
          "прошла ни одной эпохи.")

if history_path.exists():
    lines = history_path.read_text(encoding="utf-8").splitlines()
    print(f"\nИстория обучения ({len(lines)} эпох), последние 10:")
    for line in lines[-10:]:
        rec = json.loads(line)
        miou = rec.get("val_miou")
        c_miou = rec.get("val_clinical_miou")
        miou_str = f"{miou:.4f}" if miou is not None else "n/a"
        c_miou_str = f"{c_miou:.4f}" if c_miou is not None else "n/a"
        train_s = rec.get("train_seconds")
        train_s_str = f"{train_s:.0f}s" if train_s is not None else "n/a"
        print(f"  epoch {rec['epoch']:>3}  train_loss={rec['train_loss']:.4f}  "
              f"val_mIoU={miou_str}  clinical_mIoU={c_miou_str}  epoch_time={train_s_str}")


In [ ]:
# 8. Визуализация: маска сегментации, наложенная на 2-3 препарата из
# валидационной выборки — то, что пойдёт в демо. Показывается прямо здесь и
# сохраняется на Drive (заберите оттуда на локальную машину, чтобы
# посмотреть/переслать) — drive.google.com -> histovision/previews/.
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

from src.inference import segmenter as segmenter_module
from src.inference.segmenter import PatchSegmenter
from src.training.dataset import list_samples
from src.utils.bcss_classes import load_bcss_classes
from src.utils.config import get_settings

# Модель кэшируется по пути к весам на уровне процесса (src/inference/segmenter.py)
# — если эта ячейка уже запускалась в этом сеансе до того, как обучение
# сохранило более новый чекпоинт по тому же пути, кэш нужно сбросить, иначе
# здесь молча покажутся старые веса.
segmenter_module._MODEL_CACHE.clear()

settings = get_settings()
settings.segmentation_model.weights_path = "/content/drive/MyDrive/histovision/checkpoints/segmentation_best.pth"
settings.segmentation_model.device = "cuda" if torch.cuda.is_available() else "cpu"

segmenter = PatchSegmenter(settings)
taxonomy = load_bcss_classes()

palette = np.zeros((256, 3), dtype=np.uint8)
for c in taxonomy.classes:
    palette[c.model_index] = [int(c.color[i:i + 2], 16) for i in (1, 3, 5)]

val_samples = list_samples(Path("data/bcss"), "val")
preview_samples = val_samples[:3]
preview_dir = Path("/content/drive/MyDrive/histovision/previews")
preview_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, len(preview_samples), figsize=(6 * len(preview_samples), 6))
axes = [axes] if len(preview_samples) == 1 else axes

for ax, sample in zip(axes, preview_samples):
    image = Image.open(sample.image_path).convert("RGB")
    result = segmenter.segment(image)

    mask_rgb = palette[result.mask]
    resized_image = np.array(image.resize((result.mask.shape[1], result.mask.shape[0])))
    overlay = (0.55 * mask_rgb + 0.45 * resized_image).astype(np.uint8)

    out_path = preview_dir / f"{sample.id}_overlay.png"
    Image.fromarray(overlay).save(out_path)
    print(f"Сохранено: {out_path}  (mean_confidence={result.mean_confidence:.3f})")

    ax.imshow(overlay)
    ax.set_title(f"{sample.id}\nmean_confidence={result.mean_confidence:.3f}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig(preview_dir / "_grid.png", dpi=120)
plt.show()

print(f"\nВсе превью сохранены в {preview_dir}. mean_confidence ~0.09 (случайный "
      f"уровень для 16 классов ~0.0625) означал недообучение в прошлый раз — "
      f"после полноценного обучения на всех 151 регионах ожидается заметно "
      f"выше и предсказания, сконцентрированные на осмысленных классах, а не "
      f"размазанные по всем 13-16.")
